In [1]:
import copy
import json
from pathlib import Path

import torch
from safetensors.torch import save_file
from torch import nn


SEED = 41


class Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(nn.Linear(2, 32), nn.ReLU(), nn.Linear(32, 16), nn.ReLU())
        self.output = nn.Linear(16, 2)

    def forward(self, x):
        return self.output(self.features(x))


def data(n, agreement, seed):
    """Feature 0 is a noisy signal; feature 1 is a spurious correlate."""
    g = torch.Generator().manual_seed(seed)
    y = torch.randint(0, 2, (n,), generator=g)
    sign = 2 * y.float() - 1
    agree = torch.rand(n, generator=g) < agreement
    spur_sign = sign * torch.where(agree, 1.0, -1.0)
    x = torch.stack((sign + 0.6 * torch.randn(n, generator=g),
                     1.5 * spur_sign + 0.3 * torch.randn(n, generator=g)), dim=1)
    return x, y, agree


def accuracy(model, dataset):
    x, y, agree = dataset
    with torch.no_grad():
        predictions = model(x).argmax(dim=1)
    result = {}
    for name, mask in (("overall", torch.ones_like(agree)), ("agree", agree), ("conflict", ~agree)):
        result[name] = {"n": int(mask.sum()), "accuracy": float((predictions[mask] == y[mask]).float().mean()) if mask.any() else None}
    return result


def edit(model, direction, strength):
    edited = copy.deepcopy(model)
    with torch.no_grad():
        # Change the class-1 versus class-0 logit difference by strength * (h · direction).
        edited.output.weight[1].add_(direction, alpha=strength / 2)
        edited.output.weight[0].add_(direction, alpha=-strength / 2)
    return edited


def select_strength(model, direction, validation):
    # Choose using validation only; tie-break toward the smaller edit.
    candidates = [0.0, 0.25, 0.5, 1.0, 2.0, 4.0, 8.0, 16.0]
    scores = [(s, accuracy(edit(model, direction, s), validation)["overall"]["accuracy"]) for s in candidates]
    best = max(scores, key=lambda item: (item[1], -item[0]))
    return best[0], [{"strength": s, "accuracy": a} for s, a in scores]

In [2]:
torch.set_num_threads(1)
torch.manual_seed(SEED)
train = data(4000, 0.95, 1)

In [6]:
train[2][:10]

tensor([True, True, True, True, True, True, True, True, True, True])

In [7]:
model = Classifier()
optimizer = torch.optim.Adam(model.parameters(), lr=0.003)
x, y, _ = train

In [8]:
for _ in range(250):
    optimizer.zero_grad()
    loss = nn.functional.cross_entropy(model(x), y)
    loss.backward()
    optimizer.step()

In [9]:
x[0]

tensor([0.6134, 1.6004])

In [34]:
with torch.no_grad():
    res = model(torch.tensor([0.5, -1.5]))

In [36]:
inputs = torch.tensor([
    [0.5, -1.5],
    [0.5,  1.5],
])

with torch.no_grad():
    scores = model(inputs)
    predictions = scores.argmax(dim=1)

print("scores:\n", scores)
print("predictions:", predictions)

scores:
 tensor([[ 0.8103,  0.4269],
        [-2.1286,  3.2364]])
predictions: tensor([0, 1])
